# Synthetic traffic generation

Generate normal, SYN flood and port scan examples with five numerical features. The output contains 1,750 rows.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

OUTPUT_FILE = "synthetic_network_intrusion_data.csv"
SAMPLE_FILE = "dataset_sample.csv"


In [2]:
def clip_values(data, column_limits):
    """Clip each feature to its configured range."""
    for column, (min_value, max_value) in column_limits.items():
        data[column] = data[column].clip(lower=min_value, upper=max_value)
    return data


def prepare_integer_columns(data):
    """Round connection and port counts to integers."""
    data["connections_5s"] = data["connections_5s"].round().astype(int)
    data["unique_dst_ports"] = data["unique_dst_ports"].round().astype(int)
    return data


In [3]:
def generate_normal_traffic(n_samples=1500):
    data = pd.DataFrame({
        "connections_5s": np.random.normal(20, 5, n_samples),
        "syn_percentage": np.random.normal(12, 4, n_samples),
        "avg_connection_duration": np.random.normal(10, 3, n_samples),
        "unique_dst_ports": np.random.normal(3, 1, n_samples),
        "transferred_bytes": np.random.normal(80000, 25000, n_samples)
    })

    limits = {
        "connections_5s": (1, 50),
        "syn_percentage": (0, 100),
        "avg_connection_duration": (0.5, 30),
        "unique_dst_ports": (1, 10),
        "transferred_bytes": (1000, 200000)
    }

    data = clip_values(data, limits)
    data = prepare_integer_columns(data)
    data["attack_type"] = "normal"
    data["label"] = 0
    return data


def generate_syn_flood_traffic(n_samples=125):
    data = pd.DataFrame({
        "connections_5s": np.random.normal(120, 20, n_samples),
        "syn_percentage": np.random.normal(85, 8, n_samples),
        "avg_connection_duration": np.random.normal(1.5, 0.7, n_samples),
        "unique_dst_ports": np.random.normal(4, 2, n_samples),
        "transferred_bytes": np.random.normal(12000, 6000, n_samples)
    })

    limits = {
        "connections_5s": (60, 200),
        "syn_percentage": (60, 100),
        "avg_connection_duration": (0.1, 5),
        "unique_dst_ports": (1, 15),
        "transferred_bytes": (500, 50000)
    }

    data = clip_values(data, limits)
    data = prepare_integer_columns(data)
    data["attack_type"] = "syn_flood"
    data["label"] = 1
    return data


def generate_port_scan_traffic(n_samples=125):
    data = pd.DataFrame({
        "connections_5s": np.random.normal(70, 15, n_samples),
        "syn_percentage": np.random.normal(60, 10, n_samples),
        "avg_connection_duration": np.random.normal(2.0, 1.0, n_samples),
        "unique_dst_ports": np.random.normal(40, 8, n_samples),
        "transferred_bytes": np.random.normal(8000, 4000, n_samples)
    })

    limits = {
        "connections_5s": (30, 130),
        "syn_percentage": (30, 95),
        "avg_connection_duration": (0.1, 6),
        "unique_dst_ports": (15, 70),
        "transferred_bytes": (500, 40000)
    }

    data = clip_values(data, limits)
    data = prepare_integer_columns(data)
    data["attack_type"] = "port_scan"
    data["label"] = 1
    return data


In [4]:
def generate_dataset():
    normal_data = generate_normal_traffic(1500)
    syn_flood_data = generate_syn_flood_traffic(125)
    port_scan_data = generate_port_scan_traffic(125)

    dataset = pd.concat(
        [normal_data, syn_flood_data, port_scan_data],
        ignore_index=True
    )

    dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return dataset


def validate_dataset(dataset):
    expected_columns = {
        "connections_5s",
        "syn_percentage",
        "avg_connection_duration",
        "unique_dst_ports",
        "transferred_bytes",
        "attack_type",
        "label"
    }

    assert set(dataset.columns) == expected_columns
    assert len(dataset) == 1750
    assert dataset.isna().sum().sum() == 0
    assert (dataset["label"] == 0).sum() == 1500
    assert (dataset["label"] == 1).sum() == 250
    assert (dataset["attack_type"] == "syn_flood").sum() == 125
    assert (dataset["attack_type"] == "port_scan").sum() == 125


In [5]:
dataset = generate_dataset()
validate_dataset(dataset)

dataset.to_csv(OUTPUT_FILE, index=False)
dataset.head(5).to_csv(SAMPLE_FILE, index=False)

print("Dataset je uspešno generisan.")
print("Fajl:", OUTPUT_FILE)
print("Ukupan broj primera:", len(dataset))
print("\nBroj primera po tipu saobraćaja:")
print(dataset["attack_type"].value_counts())
print("\nBroj primera po labeli:")
print(dataset["label"].value_counts())
print("\nPrvih pet redova:")
display(dataset.head())


Dataset je uspešno generisan.
Fajl: synthetic_network_intrusion_data.csv
Ukupan broj primera: 1750

Broj primera po tipu saobraćaja:
normal       1500
syn_flood     125
port_scan     125
Name: attack_type, dtype: int64

Broj primera po labeli:
0    1500
1     250
Name: label, dtype: int64

Prvih pet redova:


,connections_5s,syn_percentage,avg_connection_duration,unique_dst_ports,transferred_bytes,attack_type,label
0,20,12.965288,10.492884,3,77058.626710,normal,0
1,28,5.910563,9.978865,3,128333.993693,normal,0
2,24,16.304029,6.539948,1,71330.710581,normal,0
3,17,13.045002,9.546102,4,122338.678335,normal,0
4,109,78.283780,1.635384,6,10040.317316,syn_flood,1
